In [ ]:
import numpy as np
#import open3d as o3d
from PIL import Image
from transformers import pipeline

ImportError: libusb-1.0.so.0: cannot open shared object file: No such file or directory

In [2]:
# 1. Load the lightweight CPU-friendly depth model
pipe = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device="cpu",
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 99.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

In [3]:
# 2. Load image and predict depth
image = Image.open("Krokodil.jpg").convert("RGB")
depth_result = pipe(image)
depth_array = np.array(depth_result["depth"], dtype=np.float32)

In [4]:
depth_array

array([[ 45.,  45.,  44., ...,  56.,  53.,  51.],
       [ 45.,  45.,  44., ...,  56.,  54.,  53.],
       [ 45.,  44.,  44., ...,  57.,  56.,  55.],
       ...,
       [249., 249., 249., ..., 247., 247., 247.],
       [247., 248., 248., ..., 248., 248., 248.],
       [246., 247., 248., ..., 248., 249., 249.]],
      shape=(1066, 2000), dtype=float32)

In [ ]:
# 3. Convert RGB and Depth to an Open3D RGBD image
color_o3d = o3d.geometry.Image(np.array(image))
depth_o3d = o3d.geometry.Image(depth_array)
rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
    color_o3d, depth_o3d, convert_rgb_to_intensity=False
)

In [ ]:
# 4. Project into a 3D Point Cloud and export
w, h = image.size
intrinsics = o3d.camera.PinholeCameraIntrinsic(
    w, h, fx=500, fy=500, cx=w / 2, cy=h / 2
)
pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsics)
o3d.io.write_point_cloud("output.ply", pcd)